# MathScy Data Preparation Pipeline

This notebook handles:
1. Dataset analysis and category distribution
2. Stratified sampling across math domains
3. Theorem/conjecture extraction from LaTeX
4. LLM-powered structured knowledge extraction
5. Training data formatting for MoE fine-tuning

In [ ]:
import os
import sys
import json
import random
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from pathlib import Path

# Add scripts to path
sys.path.insert(0, '/scratch/ctoxtli/moexp/scripts')
from data_utils import (
    create_stratified_sample, get_domain_group, get_primary_category,
    MATH_DOMAIN_GROUPS, CAT_TO_DOMAIN, DATA_PATH, SAMPLE_DIR
)
from llm_utils import gemini_generate, gemini_batch_generate, THEOREM_EXTRACTION_PROMPT

# Config
with open('/scratch/ctoxtli/moexp/configs/project_config.json') as f:
    config = json.load(f)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Setup complete.')

## 1. Load and Analyze Dataset Statistics

In [ ]:
# Load pre-computed stats (from background analysis job)
stats_path = '/scratch/ctoxtli/moexp/dataset_stats.json'
if os.path.exists(stats_path):
    with open(stats_path) as f:
        stats = json.load(f)
    print(f"Total entries: {stats['total_entries']:,}")
    print(f"\nMath categories ({len([k for k in stats.get('categories', {}) if k.startswith('math.')])} math-specific):")
    math_cats = {k: v for k, v in stats.get('categories', {}).items() if k.startswith('math.')}
    for cat, count in sorted(math_cats.items(), key=lambda x: -x[1])[:30]:
        print(f"  {cat}: {count:,}")
else:
    print('Stats not yet computed. Run the dataset analysis script first.')
    print('Running quick scan (first 100K lines)...')
    from data_utils import scan_dataset_categories
    stats = scan_dataset_categories(max_lines=100000)
    print(f"Quick scan: {stats['total']} entries")

## 2. Create Stratified Sample

In [ ]:
# Create stratified sample: 100 entries per primary math category
sample_path = os.path.join(SAMPLE_DIR, 'stratified_sample_100.jsonl')

if os.path.exists(sample_path):
    print(f'Loading existing sample from {sample_path}')
    sample = []
    with open(sample_path) as f:
        for line in f:
            sample.append(json.loads(line))
    print(f'Loaded {len(sample)} entries')
else:
    sample = create_stratified_sample(
        n_per_category=100,
        output_path=sample_path,
        seed=SEED
    )

In [ ]:
# Analyze sample distribution
sample_cats = Counter(get_primary_category(e.get('categories', '')) for e in sample)
sample_domains = Counter(get_domain_group(e.get('categories', '')) for e in sample)
sample_types = Counter(e.get('type', 'unknown') for e in sample)

print('=== Sample Category Distribution ===')
for cat, count in sample_cats.most_common():
    print(f'  {cat}: {count}')

print(f'\n=== Sample Domain Groups (for MoE experts) ===')
for domain, count in sample_domains.most_common():
    print(f'  {domain}: {count}')

print(f'\n=== Sample Types ===')
for t, count in sample_types.most_common():
    print(f'  {t}: {count}')

## 3. LLM-Powered Knowledge Extraction

Use Gemini to extract structured mathematical knowledge from each sample entry.

In [ ]:
def extract_knowledge_from_entry(entry: dict) -> dict:
    """Use Gemini to extract structured knowledge from a dataset entry."""
    prompt = THEOREM_EXTRACTION_PROMPT.format(
        categories=entry.get('categories', 'unknown'),
        abstract=entry.get('abstract', 'N/A')[:500],
        text=entry.get('text', '')[:2000],
        previous_context=entry.get('previous context', '')[:500],
        following_context=entry.get('following context', '')[:500],
    )
    
    response = gemini_generate(
        prompt,
        model='gemini-2.0-flash',
        temperature=0.3,
        max_tokens=2048,
    )
    
    # Try to parse JSON from response
    try:
        # Clean up response (remove markdown code blocks if present)
        clean = response.strip()
        if clean.startswith('```'):
            clean = clean.split('\n', 1)[1]
        if clean.endswith('```'):
            clean = clean.rsplit('```', 1)[0]
        return json.loads(clean)
    except json.JSONDecodeError:
        return {'raw_response': response, 'parse_error': True}

# Test with one entry
if sample:
    test_entry = sample[0]
    print(f'Testing extraction on: {test_entry.get("type", "unknown")} from {test_entry.get("categories", "unknown")}')
    print(f'Text preview: {test_entry.get("text", "")[:200]}...')
    result = extract_knowledge_from_entry(test_entry)
    print(f'\nExtracted knowledge:')
    print(json.dumps(result, indent=2)[:1000])

In [ ]:
# Batch extraction with checkpointing
extracted_path = os.path.join(SAMPLE_DIR, 'extracted_knowledge.jsonl')

# Resume from checkpoint if exists
already_processed = set()
if os.path.exists(extracted_path):
    with open(extracted_path) as f:
        for line in f:
            entry = json.loads(line)
            already_processed.add(entry.get('id', ''))
    print(f'Resuming from checkpoint: {len(already_processed)} entries already processed')

# Process remaining entries
import time
batch_size = 5
entries_to_process = [e for e in sample if e.get('id', '') not in already_processed]
print(f'Processing {len(entries_to_process)} remaining entries...')

with open(extracted_path, 'a') as outf:
    for i in range(0, len(entries_to_process), batch_size):
        batch = entries_to_process[i:i+batch_size]
        for entry in batch:
            try:
                knowledge = extract_knowledge_from_entry(entry)
                output = {
                    'id': entry.get('id', ''),
                    'categories': entry.get('categories', ''),
                    'type': entry.get('type', ''),
                    'domain_group': get_domain_group(entry.get('categories', '')),
                    'original_text': entry.get('text', ''),
                    'extracted': knowledge,
                }
                outf.write(json.dumps(output) + '\n')
                outf.flush()
            except Exception as e:
                print(f'  Error processing {entry.get("id", "unknown")}: {e}')
            time.sleep(1)  # Rate limiting
        
        if (i // batch_size + 1) % 10 == 0:
            print(f'  Processed {i + len(batch)}/{len(entries_to_process)} entries')

print('Extraction complete!')

## 4. Format Training Data for MoE

Create training examples formatted for each MoE expert domain.

In [ ]:
def format_for_moe_training(entry: dict) -> dict:
    """Format an extracted knowledge entry for MoE training."""
    extracted = entry.get('extracted', {})
    domain = entry.get('domain_group', 'other')
    
    # Create instruction-response pairs for different tasks
    examples = []
    
    # Task 1: Theorem understanding
    if extracted.get('formal_statement'):
        examples.append({
            'task': 'theorem_understanding',
            'domain': domain,
            'instruction': f'Explain the following mathematical statement in plain language:\n\n{extracted["formal_statement"]}',
            'response': extracted.get('informal_description', ''),
        })
    
    # Task 2: Conjecture generation context
    if extracted.get('potential_generalizations'):
        examples.append({
            'task': 'conjecture_generation',
            'domain': domain,
            'instruction': f'Given the following result in {domain}:\n\n{extracted.get("formal_statement", entry.get("original_text", "")[:500])}\n\nSuggest possible generalizations or related conjectures.',
            'response': extracted.get('potential_generalizations', ''),
        })
    
    # Task 3: Domain classification
    examples.append({
        'task': 'domain_classification',
        'domain': domain,
        'instruction': f'Classify the following mathematical statement into its primary domain:\n\n{entry.get("original_text", "")[:500]}',
        'response': f'This statement belongs to the domain of {domain}. Key concepts: {", ".join(extracted.get("key_concepts", []))}.',
    })
    
    return examples

# Process all extracted knowledge
training_data_path = os.path.join(SAMPLE_DIR, 'moe_training_data.jsonl')

if os.path.exists(extracted_path):
    all_training_examples = []
    with open(extracted_path) as f:
        for line in f:
            entry = json.loads(line)
            examples = format_for_moe_training(entry)
            all_training_examples.extend(examples)
    
    # Save training data
    with open(training_data_path, 'w') as f:
        for ex in all_training_examples:
            f.write(json.dumps(ex) + '\n')
    
    # Stats
    domain_counts = Counter(ex['domain'] for ex in all_training_examples)
    task_counts = Counter(ex['task'] for ex in all_training_examples)
    
    print(f'Total training examples: {len(all_training_examples)}')
    print(f'\nBy domain:')
    for d, c in domain_counts.most_common():
        print(f'  {d}: {c}')
    print(f'\nBy task:')
    for t, c in task_counts.most_common():
        print(f'  {t}: {c}')
else:
    print('Run extraction step first.')

## 5. Save checkpoint for resumability

In [ ]:
checkpoint = {
    'sample_path': sample_path,
    'extracted_path': extracted_path,
    'training_data_path': training_data_path,
    'sample_size': len(sample) if sample else 0,
    'status': 'data_preparation_complete',
}

with open(os.path.join(SAMPLE_DIR, 'data_prep_checkpoint.json'), 'w') as f:
    json.dump(checkpoint, f, indent=2)

print('Data preparation checkpoint saved.')
print('Next: Run notebook 02_moe_training.ipynb for model training.')